# Distributed Task Processing with TaskVine

This is a starter notebook for a Floability backpack. It demonstrates:
1. Connecting to a TaskVine manager
2. Defining a distributed worker function
3. Submitting and collecting tasks

**To customize this notebook:**
- Edit the `worker_function()` to implement your logic
- Modify the task parameters or task generation loop
- Read the [TaskVine documentation](https://taskvine.readthedocs.io/en/latest/) for more advanced features and configurations

## Setup: Manager Connection

TaskVine requires a manager process. The manager name and ports come from environment variables set by Floability.

In [ ]:
import os
import ndcctools.taskvine as vine

def parse_ports(ports_str: str) -> list[int]:
    ports = [int(p.strip()) for p in ports_str.split(",") if p.strip()]

    if not ports:
        raise ValueError("No valid ports provided")

    return ports

name = os.environ.get("VINE_MANAGER_NAME")

# Get manager info from environment (set by Floability)
manager_name = os.environ.get('VINE_MANAGER_NAME')
manager_ports = parse_ports(os.environ.get("VINE_MANAGER_PORTS", "9123,9150"))

print(f'Manager Name: {manager_name}')
print(f'Manager Ports: {manager_ports}')

m = vine.Manager(manager_ports, name=manager_name)
print(f"[manager] Listening on port {m.port}")

## Define Worker Function

The worker function will run on distributed worker nodes. In this example, it doubles a number and sleeps briefly.

**To customize:** Replace the function body with your processing logic.

In [ ]:
import time

def worker_function(value, sleep_time=1):
    """Double a number after simulating work."""
    time.sleep(sleep_time)
    return {"input": value, "output": value * 2}

# Test the function locally
print('Worker function defined.')
print(f'Test: worker_function(5) = {worker_function(5, sleep_time=0)}')

## Submit Tasks to Workers

This cell submits tasks to the TaskVine workers. Each task will be processed by a worker node.

**To customize:** Modify the task generation loop to match your use case.

In [ ]:
NUM_TASKS = 20
task_map = {}

for i in range(NUM_TASKS):
    task = vine.PythonTask(worker_function, i, sleep_time=1)
    task_id = m.submit(task)
    task_map[task_id] = i

print(f"[manager] Submitted {len(task_map)} tasks")

## Collect Results

Wait for tasks to complete and collect results.

In [ ]:
results = []

while not m.empty():
    done = m.wait(5)
    if not done:
        continue
    if done.successful():
        r = done.output
        r["task_id"] = done.id
        results.append(r)
        print(f"  Task {done.id}: {r['input']} → {r['output']}  worker={done.addrport}")
    else:
        print(f"  Task {done.id} failed: {done.result}")

print(f"\n[manager] All {len(results)} tasks complete.\n")